# 排序算法
在本notebook中，我们将分别介绍以下几种经典的排序算法：冒泡排序、选择排序、插入排序、希尔排序、归并排序以及快速排序。我们将分别介绍这些排序算法的思想并实现，同时，我们也会具体分析它们的算法复杂度。

## 冒泡排序
冒泡排序的核心思想是：反复比较相邻的两个元素，如果顺序不对就交换。每一轮都会把当前最大的元素“冒泡”到最后。

冒泡排序的空间复杂度为 $O(1)$ 。在不做优化的情况下，冒泡排序的时间复杂度为 $O(N^2)$ 。如果加入判断，当数组已经有序时就不再进行冒泡，此时时间复杂度可以得到优化：当数组已经有序时，时间复杂度为 $O(N)$ 。此时最坏时间复杂度、平均时间复杂度仍为 $O(N^2)$ 。

冒泡排序是稳定的。

In [12]:
def bubbleSort(nums: list[int]) -> None:
    for i in range(len(nums)):
        for j in range(0, len(nums) - i - 1):
            if nums[j] > nums[j + 1]:
                nums[j], nums[j + 1] = nums[j + 1], nums[j]

def modifiedBubbleSort(nums: list[int]) -> None:
    for i in range(len(nums)):
        swapped = False
        for j in range(0, len(nums) - i - 1):
            if nums[j] > nums[j + 1]:
                nums[j], nums[j + 1] = nums[j + 1], nums[j]
                swapped = True
        if not swapped:
            break


nums = [5, 1, 3, 6, 4]
bubbleSort(nums)
print(nums)

[1, 3, 4, 5, 6]


## 选择排序
选择排序的核心思想是：每一轮从未排序部分中找出最小的元素，放到当前未排序部分的最前面。

选择排序的空间复杂度为 $O(1)$ 。时间复杂度为 $O(N^2)$。

注意，选择排序是不稳定的。原因是：它每一轮会把“未排序部分的最小值”和当前位置的元素直接交换，这个交换可能会把某个相同元素的相对顺序打乱。以下是一个具体的反例：
```python
[2a, 2b, 1]
```
经过一次选择交换后变为
```python
[1, 2b, 2a]
```
此时列表已经有序，但是两个2之间的相对位置已经被打乱了。

In [13]:
def selectionSort(nums: list[int]) -> None:
    for i in range(len(nums) - 1):
        idx_of_smallest = i
        
        for j in range(i, len(nums)):
            if nums[j] < nums[idx_of_smallest]:
                idx_of_smallest = j
        
        nums[i], nums[idx_of_smallest] = nums[idx_of_smallest], nums[i]

nums = [5, 1, 3, 6, 4]
selectionSort(nums)
print(nums)

[1, 3, 4, 5, 6]


## 插入排序
插入排序的核心思想与我们平时玩纸牌时对纸牌排序的思路是一致的：维持一个已经排好序的子列表，将这个列表置于整个列表的前端；依次将子列表之外的元素插入到子列表中，逐渐扩大有序的子列表，直至所有元素都被扩展到子列表中。

由于不需要额外的空间，插入排序的空间复杂度是 $O(1)$ 。在最坏的情况下，即列表逆序的时候，每个元素在插入到子列表的过程中，需要与其中的每个元素发生换序，因此最坏时间复杂度是 $O(N^2)$。相反地，如果列表已经排好了序，那么时间复杂度降为 $O(N)$。平均情况下，时间复杂度为 $O(N^2)$。

插入排序是稳定的。

In [14]:
def insertionSort(nums: list[int]) -> None:
    for i in range(1, len(nums)):
        for j in reversed(range(i)):
            if nums[j + 1] < nums[j]:
                nums[j + 1], nums[j] = nums[j], nums[j + 1]
            else:
                break

nums = [5, 1, 3, 6, 4]
insertionSort(nums)
print(nums)

# 当然，还可以做进一步的优化
def modifiedInsertionSort(nums: list[int]) -> None:
    for i in range(1, len(nums)):
        cur = nums[i]
        j = i

        while j >= 1 and nums[j - 1] > cur:
            nums[j] = nums[j - 1]
            j -= 1
            
        nums[j] = cur

nums = [5, 1, 3, 6, 4]
modifiedInsertionSort(nums)
print(nums)


[1, 3, 4, 5, 6]
[1, 3, 4, 5, 6]


## 希尔排序

希尔排序可以理解为：插入排序的升级版。

普通插入排序每次只能把元素一点点往前移动，遇到很小的元素在数组后面时，需要移动很多次。希尔排序的思路是：先让元素“大步移动”，再逐渐缩小步长，最后变成普通插入排序。

具体而言，我们会预先设定一组递减到1的gaps序列。对于gap序列中的每一个gap值，我们将待排序数组切分为若干个子数组，其中每段在原始数组中间隔为gap。然后在每一个子数组内分别进行插入排序。当gap取1时，这时的插入排序相当于普通的插入排序，但是，由于数组已经大致有序了，此时再进行插入排序会快很多。

希尔排序的空间复杂度为 $O(1)$ 。希尔排序的时间复杂度依赖于gap序列的选取，通常认为其时间复杂度介于 $O(N^{1.3})\sim O(N^{2})$ 之间，最坏情况下退化为 $O(N^2)$。

希尔排序是不稳定的排序。

In [15]:
def shellSort(nums: list[int], gaps: list[int] | None = None) -> None:
    if gaps is None:
        gaps = []
        n = len(nums) // 2
        while n > 0:
            gaps.append(n)
            n //= 2

    for gap in gaps:
        for i in range(gap, len(nums)):
            cur = nums[i]
            j = i

            while j >= gap and nums[j - gap] > cur:
                nums[j] = nums[j - gap]
                j -= gap

            nums[j] = cur


nums = [5, 1, 3, 6, 9]
shellSort(nums)
print(nums)


[1, 3, 5, 6, 9]


## 归并排序

归并排序是一种经典的分治排序算法。它的核心思想是：把一个大数组不断拆成更小的子数组，直到每个子数组只剩一个元素；然后再把这些有序的小数组逐步合并成更大的有序数组。

归并排序的核心在于将两个有序的小数组合并为一个更大的有序数组——而这可以通过双指针得到简单的实现。

归并排序的时间复杂度（最优/最差/平均）为 $O(N\log N)$：无论数组是否有序，数组都会经过大约 $\log N$次切分，而每次切分后合并两个有序数组需要经过 $O(N)$次移动，故总的时间复杂度为 $O(N\log N)$。

归并排序的实现需要利用到 $O(N)$ 长度的辅助列表，因此其空间复杂度为 $O(N)$。

归并排序是稳定的。

In [16]:
# 注意，该版本的归并排序中的切片操作会导致额外的空间消耗，使得空间复杂度超过O(N)
def mergeSort(nums: list[int]) -> list[int]:
    if len(nums) <= 1:
        return nums

    n = len(nums) // 2

    left = mergeSort(nums[:n])
    right = mergeSort(nums[n:])

    return _merge(left, right)


def _merge(left: list[int], right: list[int]) -> list[int]:
    result = []
    i, j = 0, 0
    len_left, len_right = len(left), len(right)

    while i < len_left and j < len_right:
        if left[i] <= right[j]:
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1

    while i < len_left:
        result.append(left[i])
        i += 1

    while j < len_right:
        result.append(right[j])
        j += 1

    return result


nums = [5, 1, 3, 6, 9]
nums = mergeSort(nums)
print(nums)




[1, 3, 5, 6, 9]


In [17]:
# 优化版本的归并排序通过一个辅助列表实现O(N)的空间复杂度
def modifiedMergeSort(nums: list[int]) -> None:
    if len(nums) <= 1:
        return

    temp = nums[:]
    _modifiedMergeSort(nums, 0, len(nums) - 1, temp)


def _modifiedMergeSort(nums: list[int], left: int, right: int, temp: list[int]) -> None:
    if left >= right:
        return

    mid = (left + right) // 2

    _modifiedMergeSort(nums, left, mid, temp)
    _modifiedMergeSort(nums, mid + 1, right, temp)

    # 如果左右两部分已经整体有序，直接跳过合并
    if nums[mid] <= nums[mid + 1]:
        return

    _mergeWithTemp(nums, left, mid, right, temp)


def _mergeWithTemp(nums: list[int], left: int, mid: int, right: int, temp: list[int]) -> None:
    for p in range(left, right + 1):
        temp[p] = nums[p]

    i, j = left, mid + 1

    for k in range(left, right + 1):
        if i > mid:
            nums[k] = temp[j]
            j += 1
        elif j > right:
            nums[k] = temp[i]
            i += 1
        elif temp[i] <= temp[j]:
            nums[k] = temp[i]
            i += 1
        else:
            nums[k] = temp[j]
            j += 1


nums = [5, 1, 3, 6, 9]
modifiedMergeSort(nums)
print(nums)


[1, 3, 5, 6, 9]


## 快速排序

快速排序是一种经典的分治排序算法。
它的核心思想是：
选一个元素作为“基准值”`pivot`，把数组分成两部分：
小于基准值的放左边，大于基准值的放右边，然后分别对左右两部分继续排序。

快速排序的性能取决于`pivot`的选取：如果`pivot`差不多位于数组中间时，它能将数组平均切分为两半。此时的最优时间复杂度为 $O(N\log N)$。但是，如果`pivot`接近数组的最值，切分操作就白做了，因此最坏时间复杂度通常会退化到 $O(N^2)$。
快速排序的平均复杂度为 $O(N\log N)$。

快速排序的空间复杂度取决于调用栈的深度。最优能达到 $O(\log N)$，最差情况退化到 $O(N)$。

快速排序通常是非稳定的排序。

In [18]:
# 以下的实现中使用了额外的数组，空间复杂度超过O(log N)
def quickSort(nums: list[int]) -> list[int]:
    if len(nums) <= 1:
        return nums

    pivot_idx = len(nums) // 2
    pivot = nums[pivot_idx]

    left, right = [], []

    for i in range(len(nums)):
        if i != pivot_idx:
            if nums[i] < pivot:
                left.append(nums[i])
            else:
                right.append(nums[i])

    return quickSort(left) + [pivot] + quickSort(right)


nums = [5, 1, 3, 6, 9]
print(quickSort(nums))

[1, 3, 5, 6, 9]


In [19]:
# 原地排序的优化版快速排序可以将平均空间复杂度降低到O(log N)
def modifiedQuickSort(nums: list[int]) -> None:
    _modifiedQuickSort(nums, 0, len(nums) - 1)


def _modifiedQuickSort(nums: list[int], left: int, right: int) -> None:
    if left >= right:
        return

    pivot_idx = _partition(nums, left, right)

    _modifiedQuickSort(nums, left, pivot_idx - 1)
    _modifiedQuickSort(nums, pivot_idx + 1, right)


def _partition(nums: list[int], left: int, right: int) -> int:
    pivot = nums[right]
    i = left - 1

    for j in range(left, right):
        if nums[j] <= pivot:
            i += 1
            nums[i], nums[j] = nums[j], nums[i]

    nums[i + 1], nums[right] = nums[right], nums[i + 1]

    return i + 1


nums = [6, 3, 8, 5, 2, 7, 4]
modifiedQuickSort(nums)
print(nums)


[2, 3, 4, 5, 6, 7, 8]
